# DSN 2026 LID — Kaggle v2, supplied starting kit integrated

**Separate restructured copy.** Based on the full source and saved outputs of
`dsn_lid_review_safe_copy (4).ipynb`, including the completed Colab comparison.
No model training, notebook execution, account access or publishing was performed
while generating this file. The original notebook is unchanged.

## Why this version is different
- **One task selector (`STAGE`)**, not many interacting run/skip switches.
- **Run All from the top** after selecting a task. Each task loads its own
  checkpoint/data prerequisites; no dependency on a previous Colab session.
- No Drive mounts, `/content` paths, login widgets, duplicate debug cells or
  forward references. Kaggle inputs are read-only; outputs go to a fixed
  `/kaggle/working/dsn_lid_review` workspace.
- Baseline fingerprints are checked against the epoch-1 files you already loaded.
- Supplied starting-kit scripts are embedded verbatim as text and run only in
  explicit checker/packaging tasks. The kit does **not** include the inference runner.
- Local exploratory evaluation is explicitly separated from confirmed-runner evaluation.
- Results are saved immediately. Short training branches save candidate weights
  at validation checkpoints, independently of notebook output text.
- Kaggle hardware/software are recorded. Colab timings/scores are historical,
  not a promise of exact Kaggle reproduction or official compliance.

**Default `STAGE="preflight"` does not download data, load a model, train or publish.**
It checks the environment and lists candidate input folders.


## Before opening a GPU session

1. **Recover the actual epoch-1 files from your own Drive.** An ended Colab GPU
   session does not mean Drive files are deleted. This `.ipynb` does not contain
   the trained weights and cannot reconstruct them from saved outputs.
2. Attach a **private Kaggle input** containing these original files together:
   `config.json`, `preprocessor_config.json`, `model.safetensors`.
   Retain any applicable license/attribution material. Do not publish competition
   data or model artifacts unintentionally while transferring them.
3. In Kaggle, select an available GPU accelerator. Availability/quota is account
   dependent. This notebook uses **one GPU (`cuda:0`)**, even if two are visible.
4. Enable Internet when downloading packages or the official HF dataset.
   If the dataset needs authentication, add/enable a Kaggle secret named
   **`HF_TOKEN`** and set `USE_KAGGLE_HF_SECRET=True`. Grant access to the official
   dataset through HF if required. Never put a token in notebook source/output.
5. Run `preflight`; copy the correct discovered folder into `BASELINE_DIR` if
   there are multiple model folders. Nothing is silently chosen among candidates.
6. Start with `STAGE="baseline"` or `"pilot"`, not training. Inspect the starting
   kit before confirming deployment settings.

The official dataset is pinned to
`87cf6d4b1c7599f97f44b6e6aade86438369a2b1`.
No external training dataset is introduced.


## 1. Single configuration — edit here only

After editing, **Run All from the top**. `STAGE` performs one task:

| Stage | Action |
|---|---|
| `preflight` | Local environment/input checks only |
| `baseline` | Inspect and fingerprint the attached epoch-1 files |
| `pilot` | Load/audit official data and evaluate 64 full clips |
| `compare` | Repeat A/B/C on Kaggle, saving reports and predictions |
| `reference` | Establish the explicitly local or actually confirmed-runner baseline |
| `train_low_lr` | Short branch from epoch 1, original sampler |
| `train_fixed_share` | Matched short branch from epoch 1, changed sampler |
| `export` | Save a selected model into a new clean folder |
| `offline_smoke` | Fresh-process, offline, CPU functional load test |
| `check_folder` | Run the supplied official folder checker on the final export |
| `package` | Write documented submission JSON/ZIP after manual checks |

Training may use an explicitly **exploratory local** evaluation policy without
claiming it matches the hidden runner. To claim runner parity, choose
`EVALUATION_MODE="confirmed_runner"` and actually confirm its settings first.
If the real runner cannot be expressed here, adapt the helper rather than
asserting parity. Local diagnostics remain Nupe-incomplete.


In [70]:
from pathlib import Path
import os


STAGE = "preflight"
INSTALL_DEPENDENCIES = False






CFG = {
    # Empty means discover only when exactly one complete checkpoint exists.
    "BASELINE_DIR": "/kaggle/input/datasets/akinwandealex/dsn-2026-lid-epoch-1-checkpoint/epoch-1",
    "INPUT_ROOT": "/kaggle/input",
    "OUTPUT_ROOT": "/kaggle/working/dsn_lid_review",
    # Optional attached private output root from an earlier Kaggle session.
    # Read-only reference/control records can be reused only on an exact match.
    "PREVIOUS_OUTPUT_ROOT": "",
    # Temporary cache is not a model-repository artifact and may disappear.
    "HF_CACHE_DIR": "/tmp/dsn_lid_hf_cache",
    "USE_KAGGLE_HF_SECRET": True,
    "HF_SECRET_NAME": "HF_TOKEN",
    "DATASET_REPO": "Data-Science-Nigeria/dsn-2026-lid-train",
    "DATASET_COMMIT": "87cf6d4b1c7599f97f44b6e6aade86438369a2b1",
    "SEED": 42,
    "SAMPLE_RATE": 16000,
    "TRAIN_SECONDS": 8,
    "TRAIN_BATCH_SIZE": 4,
    "NUM_WORKERS": 0,  # Safer Kaggle default; identical in both short branches.
    "MAX_TRAIN_STEPS": 800,
    "EVAL_STEPS": [400, 800],
    "LR": 5e-6,
    "SCREENING_MARGIN": 0.01,  # Heuristic, not statistical significance.
    "REPRODUCTION_TOLERANCE": 0.003,  # Stop/review larger baseline discrepancies.
    # The supplied kit has no inference runner. Default is honest local research.
    "EVALUATION_MODE": "exploratory",  # or "confirmed_runner" after verification.
    "LOCAL_POLICY": "full",
    "LOCAL_BATCH_SIZE": 1,
    "LOCAL_PRECISION": "amp_fp16",
    "RUNNER_CONFIRMED": False,
    "RUNNER_POLICY": None,      # "full" or "center8" only if actually confirmed.
    "RUNNER_BATCH_SIZE": None,
    "RUNNER_PRECISION": None,   # "fp32" or "amp_fp16"; see notes below.
    # Blank selects original epoch 1. To export a branch, enter its exact path.


    "SELECTED_MODEL_DIR": "/kaggle/working/dsn_lid_review/training/20261021T122014418414Z/checkpoint-step-800",
    "ALLOW_CANDIDATE_EXPORT": True,
    "EXPORT_DTYPE": "float32",

    
    # For offline_smoke, choose an existing clean export from the printed paths.
    # "EXPORT_FOLDER": "",
    "EXPORT_FOLDER": "",
    # Set these only after inspecting/checking the actual final artifact.
    # check_folder records its result automatically against the actual folder.
    "LOCAL_RESOURCE_CHECKS_REVIEWED": False,
    "MODEL_CARD_LICENSE_REVIEWED": False,
    "PUBLIC_SNAPSHOT_CHECKED": False,
    "PUBLIC_REPO_ID": "",
    "PUBLIC_COMMIT": "",
}

# Set before HF imports. Temporary cache can be redownloaded after a new session.
os.environ["HF_HOME"] = CFG["HF_CACHE_DIR"]
os.environ["TOKENIZERS_PARALLELISM"] = "false"
assert CFG["SAMPLE_RATE"] == 16000 and CFG["TRAIN_SECONDS"] == 8
assert CFG["TRAIN_BATCH_SIZE"] == 4
assert CFG["MAX_TRAIN_STEPS"] == 800 and CFG["EVAL_STEPS"] == [400, 800]
assert CFG["LR"] == 5e-6
print("Selected task:", STAGE)
print("Outputs:", CFG["OUTPUT_ROOT"])


Selected task: export
Outputs: /kaggle/working/dsn_lid_review


## 2. Dependencies — preserve Kaggle's GPU PyTorch

First try the installed environment. If packages are missing/incompatible, enable
`INSTALL_DEPENDENCIES`, Run All, then restart the Python session if Kaggle requests
it and set installation back to False. Packages below match the successful Colab
versions where relevant; Kaggle PyTorch/CUDA are deliberately **not replaced**.
This remains an unexecuted compatibility plan, not a guarantee for every image.

Audio is decoded from the official stored bytes/path using SoundFile, **not
TorchCodec**. This avoids a common PyTorch/TorchCodec/FFmpeg version coupling.
Native decoded audio must already be finite 16 kHz mono; unexpected audio fails
loudly instead of silently changing preprocessing.


In [71]:
import importlib.metadata as im
import subprocess
import sys

if INSTALL_DEPENDENCIES:
    constraints = Path("/tmp/dsn_kaggle_torch_constraints.txt")
    keep = []
    for name in ("torch", "torchaudio", "torchvision"):
        try:
            keep.append(f"{name}=={im.version(name)}")
        except im.PackageNotFoundError:
            pass
    constraints.write_text("\n".join(keep) + "\n")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--upgrade-strategy", "only-if-needed", "-c", str(constraints),
        "transformers==5.20.1", "datasets==4.8.5",
        "huggingface-hub==1.33.0", "scikit-learn==1.6.1",
        "soundfile==0.14.0", "safetensors==0.8.0"
    ])

for name in ("torch", "transformers", "datasets", "huggingface-hub",
             "scikit-learn", "soundfile", "safetensors"):
    try:
        print(name, im.version(name))
    except im.PackageNotFoundError:
        print(name, "MISSING — install before a model/data task")


torch 2.11.0+cu128
transformers 5.16.1
datasets 4.8.5
huggingface-hub 1.29.0
scikit-learn 1.6.1
soundfile 0.14.0
safetensors 0.8.0


## 3. Historical results — not recomputed Kaggle results

The completed Colab comparison from the attached notebook is preserved below as
data. Nupe has no validation examples. These are **represented-class diagnostics**,
not official competition scores, and must not be compared with the 0.60 threshold.

| Colab evaluation | Diagnostic |
|---|---:|
| A: centered 8 seconds, batch 4, AMP fp16 | 0.563549724 |
| B: centered 8 seconds, batch 1, AMP fp16 | 0.556602978 |
| C: full clips, batch 1, AMP fp16 | 0.569405521 |

Full clips helped relative to single-clip centered crops, but Efik/Ibibio remained
at zero F1 and Igala worsened. More false Nupe predictions do not measure Nupe recall.
`ego` is **Eggon**. The checkpoint remains the 94,571,917-parameter epoch-1 model.

Historical per-class tables, confusion matrices and timings are embedded below and
saved to the output workspace during preflight. No authentication outputs/widgets
from the attached notebook are copied.


In [72]:
HISTORICAL_COLAB_RESULTS = {'A_center8_batch4': {'diagnostic': 0.5635497236298666, 'report': [{'label': 'yor', 'precision': 0.96748, 'recall': 0.721212, 'f1': 0.826389, 'support': 330}, {'label': 'ibo', 'precision': 0.907295, 'recall': 0.848011, 'f1': 0.876652, 'support': 704}, {'label': 'hau', 'precision': 0.813675, 'recall': 0.961616, 'f1': 0.881481, 'support': 495}, {'label': 'ego', 'precision': 0.65109, 'recall': 0.941441, 'f1': 0.769797, 'support': 222}, {'label': 'anw', 'precision': 0.921053, 'recall': 0.994832, 'f1': 0.956522, 'support': 387}, {'label': 'efi', 'precision': 0.028571, 'recall': 0.015625, 'f1': 0.020202, 'support': 64}, {'label': 'ibb', 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 107}, {'label': 'idu', 'precision': 0.91129, 'recall': 0.753333, 'f1': 0.824818, 'support': 450}, {'label': 'igl', 'precision': 0.080357, 'recall': 0.183673, 'f1': 0.111801, 'support': 49}, {'label': 'nup', 'precision': None, 'recall': None, 'f1': None, 'support': 0}, {'label': 'tiv', 'precision': 0.22381, 'recall': 0.220657, 'f1': 0.222222, 'support': 213}, {'label': 'urh', 'precision': 0.988479, 'recall': 0.78, 'f1': 0.871951, 'support': 550}, {'label': 'other', 'precision': 0.806157, 'recall': 0.90525, 'f1': 0.852835, 'support': 781}], 'confusion': {'yor': [238, 18, 5, 0, 1, 7, 0, 17, 13, 1, 7, 1, 22], 'ibo': [3, 597, 64, 0, 0, 0, 0, 0, 0, 0, 26, 3, 11], 'hau': [0, 11, 476, 0, 0, 0, 0, 0, 0, 0, 1, 0, 7], 'ego': [0, 0, 0, 209, 0, 0, 0, 13, 0, 0, 0, 0, 0], 'anw': [0, 0, 0, 2, 385, 0, 0, 0, 0, 0, 0, 0, 0], 'efi': [0, 11, 4, 0, 0, 1, 0, 1, 4, 10, 2, 0, 31], 'ibb': [0, 2, 1, 0, 25, 2, 0, 0, 56, 8, 1, 0, 12], 'idu': [0, 0, 0, 109, 1, 0, 0, 339, 0, 0, 1, 0, 0], 'igl': [0, 3, 1, 0, 0, 3, 0, 2, 9, 20, 2, 0, 9], 'nup': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'tiv': [1, 3, 24, 1, 3, 6, 0, 0, 21, 36, 47, 0, 71], 'urh': [0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 112, 429, 7], 'other': [4, 11, 10, 0, 3, 16, 4, 0, 9, 5, 11, 1, 707]}, 'timing': {'mean_seconds_per_clip': 0.01661895897357452, 'batch_normalized_p95': 0.02108548374997099, 'measured_pipeline_seconds': 72.3257094529963, 'cuda_allocated_peak_gb': 0.918297088, 'cuda_reserved_peak_gb': 1.224736768}, 'source': 'Saved Colab output in the attached notebook, comparison cell', 'official_score': False, 'nupe_measured': False}, 'B_center8_batch1': {'diagnostic': 0.5566029784829549, 'report': [{'label': 'yor', 'precision': 0.952191, 'recall': 0.724242, 'f1': 0.822719, 'support': 330}, {'label': 'ibo', 'precision': 0.911811, 'recall': 0.822443, 'f1': 0.864824, 'support': 704}, {'label': 'hau', 'precision': 0.816609, 'recall': 0.953535, 'f1': 0.879776, 'support': 495}, {'label': 'ego', 'precision': 0.65109, 'recall': 0.941441, 'f1': 0.769797, 'support': 222}, {'label': 'anw', 'precision': 0.910165, 'recall': 0.994832, 'f1': 0.950617, 'support': 387}, {'label': 'efi', 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 64}, {'label': 'ibb', 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 107}, {'label': 'idu', 'precision': 0.904, 'recall': 0.753333, 'f1': 0.821818, 'support': 450}, {'label': 'igl', 'precision': 0.066176, 'recall': 0.183673, 'f1': 0.097297, 'support': 49}, {'label': 'nup', 'precision': None, 'recall': None, 'f1': None, 'support': 0}, {'label': 'tiv', 'precision': 0.215859, 'recall': 0.230047, 'f1': 0.222727, 'support': 213}, {'label': 'urh', 'precision': 0.988263, 'recall': 0.765455, 'f1': 0.862705, 'support': 550}, {'label': 'other', 'precision': 0.814149, 'recall': 0.869398, 'f1': 0.840867, 'support': 781}], 'confusion': {'yor': [239, 16, 5, 0, 2, 8, 1, 20, 18, 1, 7, 0, 13], 'ibo': [9, 579, 68, 0, 0, 0, 0, 0, 0, 0, 30, 3, 15], 'hau': [0, 12, 472, 0, 0, 0, 0, 0, 0, 0, 1, 1, 9], 'ego': [0, 0, 0, 209, 0, 0, 0, 13, 0, 0, 0, 0, 0], 'anw': [0, 0, 0, 2, 385, 0, 0, 0, 0, 0, 0, 0, 0], 'efi': [0, 10, 4, 0, 2, 0, 0, 1, 6, 12, 2, 0, 27], 'ibb': [0, 0, 0, 0, 27, 1, 0, 0, 58, 9, 1, 0, 11], 'idu': [0, 0, 0, 109, 1, 0, 0, 339, 0, 0, 1, 0, 0], 'igl': [0, 3, 1, 0, 0, 4, 0, 2, 9, 21, 2, 0, 7], 'nup': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'tiv': [0, 3, 17, 1, 2, 6, 1, 0, 27, 47, 49, 0, 60], 'urh': [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 115, 421, 13], 'other': [3, 11, 11, 0, 4, 20, 5, 0, 18, 10, 19, 1, 679]}, 'timing': {'mean_seconds_per_clip': 0.021661445594212753, 'batch_normalized_p95': 0.038006491900046055, 'measured_pipeline_seconds': 94.2706112260139, 'cuda_allocated_peak_gb': 0.523500032, 'cuda_reserved_peak_gb': 1.235222528}, 'source': 'Saved Colab output in the attached notebook, comparison cell', 'official_score': False, 'nupe_measured': False}, 'C_full_batch1': {'diagnostic': 0.5694055206138771, 'report': [{'label': 'yor', 'precision': 0.96, 'recall': 0.727273, 'f1': 0.827586, 'support': 330}, {'label': 'ibo', 'precision': 0.920123, 'recall': 0.850852, 'f1': 0.884133, 'support': 704}, {'label': 'hau', 'precision': 0.834798, 'recall': 0.959596, 'f1': 0.892857, 'support': 495}, {'label': 'ego', 'precision': 0.675, 'recall': 0.972973, 'f1': 0.797048, 'support': 222}, {'label': 'anw', 'precision': 0.925659, 'recall': 0.997416, 'f1': 0.960199, 'support': 387}, {'label': 'efi', 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 64}, {'label': 'ibb', 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 107}, {'label': 'idu', 'precision': 0.930295, 'recall': 0.771111, 'f1': 0.843256, 'support': 450}, {'label': 'igl', 'precision': 0.059211, 'recall': 0.183673, 'f1': 0.089552, 'support': 49}, {'label': 'nup', 'precision': None, 'recall': None, 'f1': None, 'support': 0}, {'label': 'tiv', 'precision': 0.257143, 'recall': 0.253521, 'f1': 0.255319, 'support': 213}, {'label': 'urh', 'precision': 0.99095, 'recall': 0.796364, 'f1': 0.883065, 'support': 550}, {'label': 'other', 'precision': 0.83995, 'recall': 0.866837, 'f1': 0.853182, 'support': 781}], 'confusion': {'yor': [240, 15, 5, 0, 3, 7, 2, 17, 22, 0, 7, 0, 12], 'ibo': [8, 599, 57, 0, 0, 0, 0, 0, 0, 0, 26, 2, 12], 'hau': [0, 10, 475, 0, 0, 0, 0, 0, 0, 0, 0, 1, 9], 'ego': [0, 0, 0, 216, 0, 0, 0, 6, 0, 0, 0, 0, 0], 'anw': [0, 0, 0, 1, 386, 0, 0, 0, 0, 0, 0, 0, 0], 'efi': [0, 10, 3, 0, 2, 0, 0, 0, 7, 13, 2, 0, 27], 'ibb': [0, 0, 0, 0, 21, 0, 0, 0, 67, 9, 1, 0, 9], 'idu': [0, 0, 0, 102, 1, 0, 0, 347, 0, 0, 0, 0, 0], 'igl': [0, 3, 1, 0, 0, 4, 0, 1, 9, 26, 1, 0, 4], 'nup': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'tiv': [0, 3, 17, 1, 1, 6, 1, 2, 26, 57, 54, 0, 45], 'urh': [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 100, 438, 11], 'other': [2, 10, 11, 0, 3, 23, 5, 0, 21, 9, 19, 1, 677]}, 'timing': {'mean_seconds_per_clip': 0.03004742262752286, 'batch_normalized_p95': 0.057144932650089686, 'measured_pipeline_seconds': 130.7663832749795, 'cuda_allocated_peak_gb': 0.885408256, 'cuda_reserved_peak_gb': 1.235222528}, 'source': 'Saved Colab output in the attached notebook, comparison cell', 'official_score': False, 'nupe_measured': False}}


## Supplied starting-kit findings

The ZIP contains exactly `README.md`, example `submission.json`,
`check_model_folder.py`, and `make_submission.py`. It contains **no inference
runner or scoring implementation**. It cannot establish crop length, batch
size, autocast/dtype, exact weighting normalization, or timing details.

The checker reads filenames, config and safetensors headers only; it does not
load the model or measure runtime/memory. Its label check compares a **set**
of names, not index order. This notebook retains the stronger exact mapping
assertions. It rejects the **presence** of an `auto_map` key, even if null.

Exact permitted filenames/patterns from the supplied checker:
`config.json`, `preprocessor_config.json`, `model.safetensors`,
`model-*.safetensors`, `model.safetensors.index.json`, `README.md`, `LICENSE`,
`LICENSE.md`, `LICENSE.txt`, `.gitattributes`.
All stored tensor headers must be F32/F16. Limits are 350e6 elements and 1.5e9
bytes in its counted files. Passing it does not prove full submission compliance.

The scripts below are embedded as text from the supplied ZIP. They are not
executed by this definition cell. The example all-zero commit is not a real
submission revision. Use a real **lowercase 40-character** hash.


In [73]:
SUPPLIED_STARTING_KIT = {'README.md': '# Starting kit\n\n* `check_model_folder.py`: checks a saved model folder against the competition rules before you push it.\n* `make_submission.py`: writes `submission.json` and `submission.zip` for a Hugging Face repo.\n* `submission.json`: an example of the file you submit.\n\n```\npython check_model_folder.py ./my_model_folder\npython make_submission.py username/model-name                 # finds the latest commit on Hugging Face\npython make_submission.py username/model-name --revision <40-char hash>\n```\n\nUpload `submission.zip` on the Participate page. Only the link and commit are submitted; we download and run your model.\n', 'submission.json': '{"repo": "username/model-name", "revision": "0000000000000000000000000000000000000000"}\n', 'check_model_folder.py': '#!/usr/bin/env python\n"""\nCheck a saved model folder against the competition rules before you push it to Hugging Face.\n\n    python check_model_folder.py ./my_model_folder\n\nIt looks only at file names, config.json and the weight headers (nothing is loaded), so it is fast.\nIt cannot measure speed or memory: those are measured on the organisers\' hardware.\n"""\nimport json\nimport struct\nimport sys\nfrom fnmatch import fnmatch\nfrom pathlib import Path\n\nLABELS = ["yor", "ibo", "hau", "ego", "anw", "efi", "ibb", "idu", "igl", "nup", "tiv", "urh", "other"]\nMAX_PARAMS, MAX_BYTES = 350e6, 1.5e9\nALLOWED_DTYPES = {"F32", "F16"}\nALLOWED_FILES = ["config.json", "preprocessor_config.json", "model.safetensors", "model-*.safetensors",\n                 "model.safetensors.index.json", "README.md", "LICENSE", "LICENSE.md", "LICENSE.txt", ".gitattributes"]\nALLOWED_TYPES = {"wav2vec2", "wav2vec2-conformer", "wav2vec2-bert", "hubert", "wavlm", "unispeech", "unispeech-sat",\n                 "sew", "sew-d", "data2vec-audio", "whisper", "audio-spectrogram-transformer"}\n\n\ndef summary(path):\n    with open(path, "rb") as f:\n        n = struct.unpack("<Q", f.read(8))[0]\n        header = json.loads(f.read(n))\n    params, dtypes = 0, set()\n    for k, v in header.items():\n        if k != "__metadata__":\n            c = 1\n            for d in v["shape"]:\n                c *= d\n            params += c\n            dtypes.add(v["dtype"])\n    return params, dtypes\n\n\nfolder = Path(sys.argv[1]) if len(sys.argv) > 1 else sys.exit(__doc__)\nproblems, files = [], [p for p in folder.rglob("*") if p.is_file() and ".git/" not in p.as_posix() and ".cache" not in p.parts]\nnames = [p.relative_to(folder).as_posix() for p in files]\nfor n in names:\n    if not any(fnmatch(n, pat) for pat in ALLOWED_FILES):\n        problems.append(f"file not allowed: {n}")\nfor need in ("config.json", "preprocessor_config.json"):\n    if need not in names:\n        problems.append(f"{need} is missing")\nweights = sorted(p for p in files if fnmatch(p.name, "model*.safetensors") and p.suffix == ".safetensors")\nif not weights:\n    problems.append("no model.safetensors found (save with safe_serialization=True)")\nsize = sum(p.stat().st_size for p in files)\nif size > MAX_BYTES:\n    problems.append(f"folder is {size / 1e9:.2f} GB, limit is {MAX_BYTES / 1e9:.1f} GB")\nparams, dtypes = 0, set()\nfor w in weights:\n    p, d = summary(w)\n    params, dtypes = params + p, dtypes | d\nif params > MAX_PARAMS:\n    problems.append(f"{params / 1e6:.1f}M parameters, limit is {MAX_PARAMS / 1e6:.0f}M")\nif dtypes - ALLOWED_DTYPES:\n    problems.append(f"weight dtype {sorted(dtypes - ALLOWED_DTYPES)} not allowed; use fp32 or fp16")\nif (folder / "config.json").exists():\n    cfg = json.loads((folder / "config.json").read_text())\n    if "auto_map" in cfg:\n        problems.append("config.json has auto_map (custom code is not allowed)")\n    if cfg.get("model_type") not in ALLOWED_TYPES:\n        problems.append(f"model_type {cfg.get(\'model_type\')!r} is not supported: {sorted(ALLOWED_TYPES)}")\n    if set((cfg.get("id2label") or {}).values()) != set(LABELS):\n        problems.append(f"id2label must be exactly {LABELS}")\n\nprint(f"{params / 1e6:.1f}M parameters, {size / 1e9:.3f} GB, dtypes {sorted(dtypes)}")\nif problems:\n    print("\\nFAIL:")\n    print("\\n".join(" - " + p for p in problems))\n    sys.exit(1)\nprint("PASS: the folder meets the file, size, parameter and label rules")\n', 'make_submission.py': '#!/usr/bin/env python\n"""Create submission.json and submission.zip for a Hugging Face model repository.\n\n    python make_submission.py username/model-name                       (needs: pip install huggingface_hub)\n    python make_submission.py username/model-name --revision <40-char hash>\n"""\nimport argparse\nimport json\nimport re\nimport zipfile\n\nap = argparse.ArgumentParser()\nap.add_argument("repo", help="Hugging Face repo id, for example username/model-name")\nap.add_argument("--revision", help="40-character commit hash (default: the latest commit of the repo)")\na = ap.parse_args()\n\nif not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,95}/[A-Za-z0-9][A-Za-z0-9_.-]{0,95}", a.repo):\n    raise SystemExit("repo must look like username/model-name")\nrevision = a.revision\nif not revision:\n    from huggingface_hub import HfApi\n    revision = HfApi().model_info(a.repo).sha\nif not re.fullmatch(r"[0-9a-f]{40}", revision):\n    raise SystemExit("revision must be a full 40-character commit hash, not a branch name")\n\nwith open("submission.json", "w") as f:\n    json.dump({"repo": a.repo, "revision": revision}, f)\nwith zipfile.ZipFile("submission.zip", "w") as z:\n    z.write("submission.json")\nprint(f"Wrote submission.zip for {a.repo} @ {revision}")\n'}
SUPPLIED_KIT_SHA256 = 'b3567755272d607fb8c4aff0cd4452d191ce927b3bd21d8570d5e670bfd7592a'


## 4. Ordered helpers — definitions only

Everything below is defined before the final task dispatcher executes. No
dataset/model is loaded merely by defining these functions. Each task creates its
own context rather than expecting `valid_data` or a fingerprint in global state.


In [74]:
import gc
import hashlib
import io
import json
import random
import re
import shutil
import time
import zipfile
from datetime import datetime, timezone
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import soundfile as sf
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from datasets import load_dataset, Audio
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

LABELS = ["yor", "ibo", "hau", "ego", "anw", "efi", "ibb",
          "idu", "igl", "nup", "tiv", "urh", "other"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = dict(enumerate(LABELS))
SHARES = np.array([0.043] * 3 + [0.087] * 10, dtype=np.float64)
EXPECTED_HASHES = {
    "config.json": "abac16bf1daa454ebbffaac33e8ecfdacd01f9875ce032a3dafc9bb89c929737",
    "preprocessor_config.json": "8cdfd65ff4115423185a1512bdae100e2e0cd744f5b322417429944aaafd0827",
    "model.safetensors": "3785f80820ca8bbb2d9fc1d7f43bf7e06856987ea7ccf456f2d7b424165e2a6d",
}
SESSION = {}  # Only an optional data cache; tasks rebuild it if absent/mismatched.

def utc_id():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

def versions():
    return {p: im.version(p) for p in (
        "torch", "transformers", "datasets", "huggingface-hub",
        "scikit-learn", "soundfile", "safetensors"
    )}

def json_write(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".partial")
    tmp.write_text(json.dumps(data, indent=2, allow_nan=False) + "\n")
    tmp.replace(path)

def output_root():
    root = Path(CFG["OUTPUT_ROOT"])
    assert root.is_absolute()
    assert not root.resolve().is_relative_to(Path(CFG["INPUT_ROOT"]).resolve()),         "Outputs must not be written under read-only inputs."
    root.mkdir(parents=True, exist_ok=True)
    return root

def record_roots():
    roots = [output_root()]
    if CFG["PREVIOUS_OUTPUT_ROOT"]:
        previous = Path(CFG["PREVIOUS_OUTPUT_ROOT"])
        assert previous.is_dir(), "Configured previous output root does not exist."
        if previous.resolve() != roots[0].resolve():
            roots.append(previous)
    return roots

def new_output(kind):
    path = output_root() / kind / utc_id()
    path.mkdir(parents=True, exist_ok=False)
    return path

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def candidate_folders():
    root = Path(CFG["INPUT_ROOT"])
    if not root.is_dir():
        return []
    return sorted({
        p.parent for p in root.rglob("config.json")
        if (p.parent / "preprocessor_config.json").is_file()
        and (p.parent / "model.safetensors").is_file()
    })

def baseline_path():
    if CFG["BASELINE_DIR"].strip():
        path = Path(CFG["BASELINE_DIR"])
    else:
        choices = candidate_folders()
        if len(choices) != 1:
            raise RuntimeError(
                f"Found {len(choices)} complete model folders. Run preflight, "
                "attach epoch-1 files if missing, then set the exact BASELINE_DIR."
            )
        path = choices[0]
    assert path.is_dir(), f"Checkpoint folder missing: {path}"
    for name in EXPECTED_HASHES:
        assert (path / name).is_file(), f"Required checkpoint file missing: {name}"
    return path

def inspect_baseline(show=True):
    path = baseline_path()
    hashes = {name: sha256_file(path / name) for name in EXPECTED_HASHES}
    if hashes != EXPECTED_HASHES:
        different = [k for k in hashes if hashes[k] != EXPECTED_HASHES[k]]
        raise RuntimeError(
            f"Baseline differs from the reviewed epoch-1 files: {different}. "
            "Do not disable this check blindly; verify the selected files first."
        )
    config = json.loads((path / "config.json").read_text())
    assert "auto_map" not in config, "The supplied checker rejects any auto_map key."
    assert config["label2id"] == LABEL2ID
    assert {int(k): v for k, v in config["id2label"].items()} == ID2LABEL
    fe = AutoFeatureExtractor.from_pretrained(
        str(path), local_files_only=True, trust_remote_code=False
    )
    assert fe.sampling_rate == CFG["SAMPLE_RATE"]
    assert fe.do_normalize is True
    assert fe.return_attention_mask is False
    if show:
        print("Verified epoch-1 folder:", path)
        print("SHA256:", hashes)
        print("Feature extractor:", fe.to_dict())
    return {"path": path, "hashes": hashes, "fe": fe}

def require_gpu():
    if not torch.cuda.is_available():
        raise RuntimeError("Select a Kaggle GPU accelerator before this task.")

def load_model(path, gpu=True):
    if gpu:
        require_gpu()
    model = AutoModelForAudioClassification.from_pretrained(
        str(path), local_files_only=True, trust_remote_code=False
    ).float()
    assert model.config.num_labels == 13
    assert model.config.label2id == LABEL2ID
    assert {int(k): v for k, v in model.config.id2label.items()} == ID2LABEL
    assert "auto_map" not in model.config.to_dict(), "Forbidden auto_map key."
    return model.to("cuda:0" if gpu else "cpu").eval()

def clean_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def seed_all():
    random.seed(CFG["SEED"])
    np.random.seed(CFG["SEED"])
    torch.manual_seed(CFG["SEED"])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(CFG["SEED"])
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    seed = torch.initial_seed() % (2**32)
    random.seed(seed)
    np.random.seed(seed)

def hf_credential():
    if not CFG["USE_KAGGLE_HF_SECRET"]:
        return None  # Public access or an existing HF credential, if available.
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(CFG["HF_SECRET_NAME"])
    except Exception:
        raise RuntimeError(
            "Cannot read the enabled Kaggle HF secret. Configure it privately "
            "in your Kaggle account; never paste its value into source or chat."
        ) from None
    if not value or not value.strip():
        raise RuntimeError("Configured HF secret is empty.")
    return value.strip()

def full_indices(dataset, seed):
    labels = np.asarray(dataset["label"])
    rng = np.random.default_rng(seed)
    chosen = []
    for label in LABELS:
        idx = np.flatnonzero(labels == label)
        rng.shuffle(idx)
        chosen.extend(idx.tolist())
    rng.shuffle(chosen)
    return chosen

def speaker_keys(frame):
    part = frame.dropna(subset=["source", "speaker_id"]).copy()
    for col in ("source", "speaker_id"):
        part[col] = part[col].astype(str).str.strip()
    part = part[part["source"].ne("") & part["speaker_id"].ne("")]
    return set(zip(part["source"], part["speaker_id"]))

def ensure_data():
    key = (CFG["DATASET_REPO"], CFG["DATASET_COMMIT"], CFG["SEED"],
           CFG["HF_CACHE_DIR"])
    if SESSION.get("data_key") == key:
        return SESSION["data"]
    assert re.fullmatch(r"[0-9a-fA-F]{40}", CFG["DATASET_COMMIT"])
    token = hf_credential()
    try:
        ds = load_dataset(
            CFG["DATASET_REPO"], revision=CFG["DATASET_COMMIT"],
            token=token, cache_dir=str(Path(CFG["HF_CACHE_DIR"]) / "datasets")
        )
    except Exception as exc:
        response = getattr(exc, "response", None)
        status = getattr(response, "status_code", None)
        raise RuntimeError(
            f"Official dataset load failed: {type(exc).__name__}, HTTP {status}. "
            "Check Kaggle Internet, HF dataset permission, and enabled private "
            "HF_TOKEN secret if needed. No token value is printed."
        ) from None
    finally:
        token = None
    # Avoid TorchCodec while retaining original stored waveform bytes/path.
    ds = ds.cast_column("audio", Audio(decode=False))
    cols = ["id", "label", "label_id", "speaker_id", "source", "duration"]
    frames = {
        split: ds[split].select_columns(cols).to_pandas()
        for split in ("train", "validation")
    }
    for split, frame in frames.items():
        assert frame["id"].notna().all() and frame["id"].is_unique, split
        assert frame["label"].notna().all(), split
        assert set(frame["label"]) <= set(LABELS), split
    train, valid = frames["train"], frames["validation"]
    assert set(train["label"]) == set(LABELS)
    assert not (set(train["id"]) & set(valid["id"])), "Shared clip IDs."
    assert not (speaker_keys(train) & speaker_keys(valid)), "Known speaker overlap."
    data = {
        "train": ds["train"].select(full_indices(ds["train"], CFG["SEED"])),
        "valid": ds["validation"].select(
            full_indices(ds["validation"], CFG["SEED"] + 1)),
        "frames": frames
    }
    print("Official pinned data:", CFG["DATASET_COMMIT"])
    print("Train / validation:", len(data["train"]), len(data["valid"]))
    print("Shared IDs / known speaker pairs: 0 / 0")
    print("Nupe validation clips:", int(valid["label"].eq("nup").sum()))
    audit_dir = output_root() / "data_audit"
    audit_dir.mkdir(exist_ok=True)
    display(pd.concat({
        split: frame.groupby("label").agg(
            clips=("id", "size"), speakers=("speaker_id", "nunique"),
            hours=("duration", lambda x: x.sum() / 3600)
        ).reindex(LABELS) for split, frame in frames.items()
    }, axis=1))
    for split, frame in frames.items():
        frame.groupby("label").size().reindex(LABELS, fill_value=0).to_csv(
            audit_dir / f"{split}_counts.csv")
    json_write(audit_dir / "provenance.json", {
        "repo": CFG["DATASET_REPO"], "commit": CFG["DATASET_COMMIT"],
        "clip_overlap": 0, "known_speaker_overlap": 0,
        "decoder": "SoundFile; Audio(decode=False); strict native 16kHz mono"
    })
    SESSION.update(data_key=key, data=data)
    return data

def decode_wave(row):
    audio = row["audio"]
    if audio.get("bytes") is not None:
        source = io.BytesIO(audio["bytes"])
    elif audio.get("path"):
        source = audio["path"]
    else:
        raise RuntimeError(f"No encoded audio bytes/path for clip {row['id']}")
    try:
        wave, sr = sf.read(source, dtype="float32", always_2d=False)
    except Exception:
        raise RuntimeError(
            f"SoundFile could not decode clip {row['id']}. Inspect its encoding; "
            "do not silently change the decoder/preprocessing."
        ) from None
    if wave.ndim == 2 and wave.shape[1] == 1:
        wave = wave[:, 0]
    assert sr == 16000, (row["id"], "unexpected native sample rate", sr)
    assert wave.ndim == 1, (row["id"], "expected mono", wave.shape)
    assert len(wave) > 0 and np.isfinite(wave).all(), row["id"]
    return np.ascontiguousarray(wave, dtype=np.float32)

def context():
    require_gpu()
    return {"baseline": inspect_baseline(), "data": ensure_data()}

def eval_settings():
    mode = CFG["EVALUATION_MODE"]
    assert mode in {"exploratory", "confirmed_runner"}
    if mode == "exploratory":
        settings = dict(policy=CFG["LOCAL_POLICY"],
                        batch_size=CFG["LOCAL_BATCH_SIZE"],
                        precision=CFG["LOCAL_PRECISION"])
        print("EXPLORATORY LOCAL evaluation — hidden-runner settings unconfirmed.")
    else:
        assert CFG["RUNNER_CONFIRMED"], \
            "Obtain/inspect actual runner settings before asserting confirmation."
        settings = dict(policy=CFG["RUNNER_POLICY"],
                        batch_size=CFG["RUNNER_BATCH_SIZE"],
                        precision=CFG["RUNNER_PRECISION"])
    assert settings["policy"] in {"full", "center8"}
    assert isinstance(settings["batch_size"], int) and settings["batch_size"] > 0
    assert settings["precision"] in {"fp32", "amp_fp16"}, \
        "Native fp16 or other inference modes require adapting this helper."
    return settings

def amp_context(precision):
    assert precision in {"fp32", "amp_fp16"}
    return autocast("cuda", dtype=torch.float16) if precision == "amp_fp16" else nullcontext()

def metrics(rows):
    truth, pred = rows["truth"].to_numpy(), rows["pred"].to_numpy()
    p, r, f, n = precision_recall_fscore_support(
        truth, pred, labels=list(range(13)), zero_division=0
    )
    present = n > 0
    contribution = float(np.dot(SHARES[present], f[present]))
    report = pd.DataFrame(dict(label=LABELS, precision=p, recall=r, f1=f, support=n))
    report.loc[~present, ["precision", "recall", "f1"]] = np.nan
    cm = pd.DataFrame(
        confusion_matrix(truth, pred, labels=list(range(13))),
        index=LABELS, columns=LABELS
    )
    return {
        "diagnostic": contribution / float(SHARES[present].sum()),
        "represented_contribution": contribution,
        "represented_mass": float(SHARES[present].sum()),
        "unmeasured": [LABELS[i] for i in range(13) if not present[i]],
        "false_nupe": int(((pred == LABEL2ID["nup"]) &
                           (truth != LABEL2ID["nup"])).sum()),
        "report": report, "confusion": cm
    }

@torch.inference_mode()
def evaluate(model, ctx, policy, batch_size, precision, limit=None):
    model.eval()
    fe, dataset = ctx["baseline"]["fe"], ctx["data"]["valid"]
    assert policy in {"center8", "full"} and batch_size > 0
    warm = fe([np.zeros(8 * 16000, dtype=np.float32)],
              sampling_rate=16000, padding=True, return_tensors="pt")
    with amp_context(precision):
        model(**{k: v.to("cuda:0") for k, v in warm.items()})
    torch.cuda.synchronize()
    del warm
    torch.cuda.reset_peak_memory_stats()
    count = len(dataset) if limit is None else min(int(limit), len(dataset))
    records, elapsed_total = [], 0.0
    for first in range(0, count, batch_size):
        started = time.perf_counter()
        items, waves = [], []
        for i in range(first, min(first + batch_size, count)):
            row = dataset[i]
            wave = decode_wave(row)
            seconds = len(wave) / 16000
            if policy == "center8" and len(wave) > 8 * 16000:
                offset = (len(wave) - 8 * 16000) // 2
                wave = wave[offset:offset + 8 * 16000]
            waves.append(wave)
            items.append(dict(id=row["id"], truth=LABEL2ID[row["label"]],
                              source=row["source"], speaker_id=row["speaker_id"],
                              seconds=seconds))
        batch = fe(waves, sampling_rate=16000, padding=True, return_tensors="pt")
        inputs = {k: v.to("cuda:0") for k, v in batch.items()}
        with amp_context(precision):
            logits = model(**inputs).logits
        assert logits.shape == (len(items), 13) and torch.isfinite(logits).all()
        predictions = logits.argmax(-1).cpu().tolist()
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        elapsed_total += elapsed
        records.extend(dict(item, pred=pred, batch_amortized_s=elapsed / len(items))
                       for item, pred in zip(items, predictions))
        del batch, inputs, logits
    rows = pd.DataFrame(records)
    timing = {
        "mean_s_per_clip": elapsed_total / count,
        "batch_normalized_p95_s": float(rows["batch_amortized_s"].quantile(.95)),
        "measured_pipeline_s": elapsed_total,
        "cuda_allocated_peak_gb": torch.cuda.max_memory_allocated() / 1e9,
        "cuda_reserved_peak_gb": torch.cuda.max_memory_reserved() / 1e9,
    }
    print("Policy / batch / precision / clips:", policy, batch_size, precision, count)
    print("Native decoded duration range:", rows["seconds"].min(), rows["seconds"].max())
    print("PRELIMINARY timing/CUDA only:", timing)
    result = dict(rows=rows, timing=timing, complete=(limit is None),
                  settings=dict(policy=policy, batch_size=batch_size, precision=precision))
    if limit is None:
        result.update(metrics(rows))
        display(result["report"])
        display(result["confusion"])
        print("Represented-class diagnostic:", result["diagnostic"])
        print("Unmeasured:", result["unmeasured"])
        print("False Nupe predictions:", result["false_nupe"], "(not Nupe recall)")
    else:
        result["diagnostic"] = None
        print("Partial timing pilot only — not a selection metric.")
    return result

def signature(ctx, settings):
    return {
        "baseline_hashes": ctx["baseline"]["hashes"],
        "dataset_repo": CFG["DATASET_REPO"], "dataset_commit": CFG["DATASET_COMMIT"],
        "evaluation": settings, "seed": CFG["SEED"], "versions": versions(),
        "gpu": torch.cuda.get_device_name(0),
        "decoder": "soundfile_strict_native_16k_mono",
    }

def save_result(path, result, provenance):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    result["rows"].to_csv(path / "predictions.csv", index=False)
    if result["complete"]:
        result["report"].to_csv(path / "per_class.csv", index=False)
        result["confusion"].to_csv(path / "confusion.csv")
    summary = {
        k: v for k, v in result.items()
        if k not in {"rows", "report", "confusion"}
    }
    summary["provenance"] = provenance
    summary["evaluation_mode"] = CFG["EVALUATION_MODE"]
    summary["official_score"] = False
    summary["runner_confirmation_claimed"] = (
        CFG["EVALUATION_MODE"] == "confirmed_runner" and CFG["RUNNER_CONFIRMED"]
        and result["settings"] == dict(policy=CFG["RUNNER_POLICY"],
            batch_size=CFG["RUNNER_BATCH_SIZE"], precision=CFG["RUNNER_PRECISION"]))
    json_write(path / "summary.json", summary)
    return summary

def reference(ctx):
    settings = eval_settings()
    sig = signature(ctx, settings)
    sig["evaluation_mode"] = CFG["EVALUATION_MODE"]
    key = hashlib.sha256(json.dumps(sig, sort_keys=True).encode()).hexdigest()[:20]
    path = output_root() / "references" / key
    for root in record_roots():
        previous_path = root / "references" / key
        summary_path = previous_path / "summary.json"
        if summary_path.is_file():
            saved = json.loads(summary_path.read_text())
            assert saved["provenance"] == sig and saved["complete"]
            print("Using matching saved Kaggle reference:", previous_path)
            return saved
    model = load_model(ctx["baseline"]["path"])
    try:
        result = evaluate(model, ctx, **settings)
        saved = save_result(path, result, sig)
    finally:
        del model
        clean_gpu()
    print("Saved current-configuration LOCAL baseline:", path)
    print("Evaluation mode:", CFG["EVALUATION_MODE"], "— not an official score.")
    return saved


## 5. Local tasks and automatically persisted results

The pilot is 64 full clips. `compare` repeats original-style centered batch 4,
single-centered batch 1, and full batch 1, all AMP fp16.

These are screening measurements only: they omit CPU RAM and non-PyTorch GPU
allocations and are not median-of-three official runs. Model loading/reporting are
outside timed batches. An actual checker/runner pass is still required.

Changing decoder/software/hardware can affect reproduction. A material discrepancy
from the historical baseline is a reason to inspect the environment before training,
not automatically change masks, labels or calibration. Full clips are not assumed
to be the starting kit's actual policy.


In [75]:
def stage_preflight():
    root = output_root()
    choices = candidate_folders()
    print("Input root exists:", Path(CFG["INPUT_ROOT"]).is_dir())
    print("Complete checkpoint candidates:")
    for path in choices:
        print(" -", path)
    if not choices:
        print("NO MODEL FOUND. Attach the actual epoch-1 folder as a private input.")
    print("Configured baseline:", CFG["BASELINE_DIR"] or "(discover exactly one)")
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print("GPU", i, torch.cuda.get_device_name(i))
        print("Only cuda:0 will be used; no automatic multi-GPU/batch-size change.")
    print("Output disk free GB:", shutil.disk_usage(root).free / 1e9)
    print("HF secret mode configured:", CFG["USE_KAGGLE_HF_SECRET"])
    print("No secret fetched, data downloaded, model loaded or training performed.")
    json_write(root / "historical_colab_results.json", HISTORICAL_COLAB_RESULTS)
    json_write(root / "environment.json", {
        "versions": versions(), "gpu_available": torch.cuda.is_available(),
        "gpus": [torch.cuda.get_device_name(i)
                 for i in range(torch.cuda.device_count())],
        "data_commit": CFG["DATASET_COMMIT"],
    })

def stage_baseline():
    base = inspect_baseline()
    model = load_model(base["path"], gpu=False)
    try:
        print("Parameters:", sum(p.numel() for p in model.parameters()))
        print("In-memory inspection dtypes:", sorted({str(p.dtype) for p in model.parameters()}))
        print("Model loaded locally; no training or weight write.")
        json_write(output_root() / "verified_baseline.json", {
            "path": str(base["path"]), "hashes": base["hashes"],
            "parameters": sum(p.numel() for p in model.parameters()),
            "versions": versions()
        })
    finally:
        del model
        clean_gpu()

def stage_pilot():
    ctx = context()
    model = load_model(ctx["baseline"]["path"])
    try:
        settings = dict(policy="full", batch_size=1, precision="amp_fp16")
        result = evaluate(model, ctx, **settings, limit=64)
        path = new_output("pilots")
        save_result(path, result, signature(ctx, settings))
        print("Rough validation minutes:",
              result["timing"]["mean_s_per_clip"] * len(ctx["data"]["valid"]) / 60)
        print("Pilot saved:", path)
    finally:
        del model
        clean_gpu()

def stage_compare():
    ctx = context()
    path = new_output("comparisons")
    model = load_model(ctx["baseline"]["path"])
    scores = {}
    try:
        for name, policy, batch in (
            ("A_center8_batch4", "center8", 4),
            ("B_center8_batch1", "center8", 1),
            ("C_full_batch1", "full", 1),
        ):
            print("\n===", name, "===")
            settings = dict(policy=policy, batch_size=batch, precision="amp_fp16")
            result = evaluate(model, ctx, **settings)
            save_result(path / name, result, signature(ctx, settings))
            scores[name] = result["diagnostic"]
        print("Kaggle diagnostics:", scores)
        print("Historical Colab A:", HISTORICAL_COLAB_RESULTS["A_center8_batch4"]["diagnostic"])
        print("Full minus single-center:",
              scores["C_full_batch1"] - scores["B_center8_batch1"])
        print("Not official scores; Nupe remains unmeasured.")
        json_write(path / "comparison.json", scores)
        print("Saved reports, matrices and clip predictions:", path)
    finally:
        del model
        clean_gpu()

def stage_reference():
    eval_settings()  # Explicit local-research or actually confirmed-runner mode.
    reference(context())


## 6. Optional training — two short, controlled branches only

`train_low_lr`: start from epoch 1, fresh AdamW, LR 5e-6, original inverse-square-root
sampling. `train_fixed_share`: independently start from epoch 1, same recipe,
only change sampling to `class_share / class_count`. Do not also class-weight loss.
The fixed-share branch requires a matching saved low-LR control.

At most 800 updates (~13% of an original epoch) and validation at 400/800. Two
validation sweeps can dominate runtime; Kaggle speed depends on its allocated GPU.
Training requires a saved Kaggle `compare` run matching the current environment,
with original-style A within 0.003 of its Colab diagnostic. A larger difference
needs review; do not simply loosen the tolerance to make the task run.
Stop at 400 if diagnostic drops >0.01 below the matched local baseline.
Any saved candidate is not automatically better than epoch 1. Prefer a clear gain
supported by weak-language precision/recall, not Nupe suppression or tiny tuning gains.

Candidate checkpoints and history are saved after validation. **No optimizer
state is saved**, so this is not exact mid-training resume after a session loss.
Use Kaggle's notebook/output persistence to retain files; `/kaggle/working` alone
is not a promise of survival across sessions. Download outputs or attach a saved
private output as a new input before changing sessions. Do not publish sensitive
data or put training logs in the submission model folder.


In [76]:
class TrainRows(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, index):
        row = self.dataset[index]
        wave = decode_wave(row)
        cap = 8 * 16000
        if len(wave) > cap:
            first = random.randint(0, len(wave) - cap)
            wave = wave[first:first + cap]
        return wave, LABEL2ID[row["label"]]

class TrainCollator:
    def __init__(self, fe):
        self.fe = fe
    def __call__(self, rows):
        waves, targets = zip(*rows)
        batch = self.fe(list(waves), sampling_rate=16000,
                        padding=True, return_tensors="pt")
        batch["labels"] = torch.tensor(targets, dtype=torch.long)
        return batch

def training_recipe(ctx):
    return {
        "evaluation_mode": CFG["EVALUATION_MODE"],
        "evaluation_signature": signature(ctx, eval_settings()),
        "seed": CFG["SEED"], "lr": CFG["LR"],
        "batch": CFG["TRAIN_BATCH_SIZE"], "workers": CFG["NUM_WORKERS"],
        "steps": CFG["MAX_TRAIN_STEPS"], "eval_steps": CFG["EVAL_STEPS"],
        "training_crop_seconds": 8, "weight_decay": 0.01,
        "training_precision": "amp_fp16",
    }

def matching_controls(recipe):
    found = []
    for root in record_roots():
        for p in (root / "training").glob("*/run.json"):
            saved = json.loads(p.read_text())
            if (saved.get("sampling") == "sqrt" and saved.get("recipe") == recipe
                    and saved.get("history")):
                found.append(p)
    return sorted(found)

def require_reproduced_baseline(ctx):
    settings = dict(policy="center8", batch_size=4, precision="amp_fp16")
    expected_signature = signature(ctx, settings)
    for root in record_roots():
        for p in sorted((root / "comparisons").glob("*/A_center8_batch4/summary.json"),
                        reverse=True):
            saved = json.loads(p.read_text())
            if saved.get("provenance") == expected_signature and saved.get("complete"):
                historical = HISTORICAL_COLAB_RESULTS["A_center8_batch4"]["diagnostic"]
                difference = abs(saved["diagnostic"] - historical)
                assert difference <= CFG["REPRODUCTION_TOLERANCE"],                     ("Kaggle reproduction differs materially; inspect before training.",
                     saved["diagnostic"], historical)
                print("Matching Kaggle baseline reproduction:", p)
                return
    raise RuntimeError(
        "Run STAGE='compare' once on this Kaggle environment before training. "
        "A matching saved comparison can also come from PREVIOUS_OUTPUT_ROOT."
    )

def stage_train(sampling):
    settings = eval_settings()
    assert sampling in {"sqrt", "fixed_share"}
    ctx = context()
    require_reproduced_baseline(ctx)
    recipe = training_recipe(ctx)
    if sampling == "fixed_share":
        controls = matching_controls(recipe)
        assert controls, "Run the matched train_low_lr control first."
        print("Matched control:", controls[-1])
    baseline = reference(ctx)  # Computes/saves it if no matching reference exists.
    baseline_score = baseline["diagnostic"]
    path = new_output("training")
    state = {
        "sampling": sampling, "recipe": recipe,
        "baseline_diagnostic": baseline_score, "history": [],
        "best_candidate_dir": None, "best_candidate_diagnostic": None,
        "optimizer_state_saved": False,
    }
    json_write(path / "run.json", state)
    seed_all()
    train = ctx["data"]["train"]
    labels = np.asarray(train["label"])
    counts = pd.Series(labels).value_counts()
    if sampling == "sqrt":
        weights = [1 / np.sqrt(counts[x]) for x in labels]
    else:
        weights = [SHARES[LABEL2ID[x]] / counts[x] for x in labels]
    sampler = WeightedRandomSampler(
        torch.tensor(weights, dtype=torch.double),
        num_samples=len(weights), replacement=True,
        generator=torch.Generator().manual_seed(CFG["SEED"])
    )
    loader = DataLoader(
        TrainRows(train), batch_size=CFG["TRAIN_BATCH_SIZE"], sampler=sampler,
        collate_fn=TrainCollator(ctx["baseline"]["fe"]),
        num_workers=CFG["NUM_WORKERS"], pin_memory=True,
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(CFG["SEED"] + 100)
    )
    model = load_model(ctx["baseline"]["path"])
    model.freeze_feature_encoder()
    model.gradient_checkpointing_enable()
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad],
                      lr=CFG["LR"], weight_decay=0.01)
    scaler = GradScaler("cuda")
    losses = []
    started = time.perf_counter()
    model.train()
    try:
        for step, batch in enumerate(loader, 1):
            targets = batch.pop("labels").to("cuda:0")
            inputs = {k: v.to("cuda:0", non_blocking=True) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with autocast("cuda", dtype=torch.float16):
                loss = model(**inputs, labels=targets).loss
            assert torch.isfinite(loss), "Nonfinite loss; stop this branch."
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach()))
            del loss, inputs, targets, batch
            if step % 100 == 0:
                print("step", step, "recent loss", np.mean(losses[-100:]))
            if step in CFG["EVAL_STEPS"]:
                optimizer.zero_grad(set_to_none=True)
                result = evaluate(model, ctx, **settings)
                save_result(path / f"step-{step}", result, recipe["evaluation_signature"])
                score = result["diagnostic"]
                entry = dict(step=step, diagnostic=score,
                             delta_from_baseline=score - baseline_score,
                             false_nupe=result["false_nupe"],
                             elapsed_minutes=(time.perf_counter() - started) / 60)
                state["history"].append(entry)
                if (state["best_candidate_diagnostic"] is None or
                        score > state["best_candidate_diagnostic"]):
                    target = path / f"checkpoint-step-{step}"
                    temporary = path / f"checkpoint-step-{step}.partial"
                    assert not target.exists() and not temporary.exists()
                    model.save_pretrained(str(temporary), safe_serialization=True)
                    ctx["baseline"]["fe"].save_pretrained(str(temporary))
                    temporary.rename(target)
                    state["best_candidate_dir"] = str(target)
                    state["best_candidate_diagnostic"] = score
                json_write(path / "run.json", state)
                pd.DataFrame(state["history"]).to_csv(path / "history.csv", index=False)
                if step == CFG["EVAL_STEPS"][0] and score < baseline_score - CFG["SCREENING_MARGIN"]:
                    print("Early stop: diagnostic fell by more than 0.01.")
                    break
                model.train()
            if step >= CFG["MAX_TRAIN_STEPS"]:
                break
    finally:
        del model, optimizer, scaler
        clean_gpu()
        assert inspect_baseline(show=False)["hashes"] == ctx["baseline"]["hashes"]
    display(pd.DataFrame(state["history"]))
    print("Independent branch saved:", path)
    print("Candidate (not automatically selected):", state["best_candidate_dir"])


## 7. Clean export, offline smoke and submission packaging

These are local operations only. There is **no HF publishing or Codabench upload
code**. `SELECTED_MODEL_DIR=""` exports epoch 1; a candidate requires its exact
folder and explicit `ALLOW_CANDIDATE_EXPORT=True`.

`export` creates a new folder and prints its path. Put that path in `EXPORT_FOLDER`
for `offline_smoke`. Add a reviewed README/license/attribution afterwards and run
the official `check_model_folder.py` on the entire final folder. Do not guess its
arguments. The supplied kit also provides `make_submission.py`. This copy embeds both
scripts verbatim; `check_folder` runs the checker, and `package` invokes the
packager with an explicit pinned commit. Embedded scripts are materialized
**outside** the model folder. Do not upload them to the HF repository.

The offline smoke is a fresh-process CPU functional check using a synthetic
two-second input. fp16 exports are converted to fp32 *in memory* for that CPU test.
It does not establish native inference dtype compatibility, real-audio accuracy,
or the official GPU/RAM/time limits.


In [77]:
def folder_hashes(path):
    path = Path(path)
    return {str(p.relative_to(path)): sha256_file(p)
            for p in sorted(path.rglob("*")) if p.is_file()}

def stage_export():
    base = inspect_baseline()
    selected = Path(CFG["SELECTED_MODEL_DIR"]) if CFG["SELECTED_MODEL_DIR"] else base["path"]
    assert selected.is_dir()
    if selected.resolve() != base["path"].resolve():
        assert CFG["ALLOW_CANDIDATE_EXPORT"], "Explicitly authorize candidate selection."
    assert CFG["EXPORT_DTYPE"] in {"float32", "float16"}
    path = new_output("exports")
    model_dir = path / "model"
    model = load_model(selected, gpu=False)
    try:
        model.gradient_checkpointing_disable()
        model = model.float() if CFG["EXPORT_DTYPE"] == "float32" else model.half()
        assert {p.dtype for p in model.parameters()} <= {torch.float32, torch.float16}
        assert sum(p.numel() for p in model.parameters()) <= 350_000_000
        model.save_pretrained(str(model_dir), safe_serialization=True, max_shard_size="1GB")
        fe = AutoFeatureExtractor.from_pretrained(
            str(selected), local_files_only=True, trust_remote_code=False)
        fe.save_pretrained(str(model_dir))
        files = [p for p in model_dir.rglob("*") if p.is_file()]
        assert {p.name for p in files} == {
            "model.safetensors", "config.json", "preprocessor_config.json"
        }, "Unexpected export files; inspect before publishing."
        assert sum(p.stat().st_size for p in files) < 1_500_000_000
        json_write(path / "export_metadata.json", {
            "selected_checkpoint": str(selected), "dtype": CFG["EXPORT_DTYPE"],
            "baseline_hashes": base["hashes"], "files_sha256": folder_hashes(model_dir),
            "versions": versions(), "offline_smoke_passed": False,
        })
        print("Clean model folder:", model_dir)
        print("Set CFG['EXPORT_FOLDER'] to that path for offline_smoke.")
        print("README/license review and STAGE=check_folder still required.")
    finally:
        del model
        clean_gpu()

def selected_export():
    assert CFG["EXPORT_FOLDER"], "Set the exact export model folder printed by export."
    path = Path(CFG["EXPORT_FOLDER"])
    assert (path / "model.safetensors").is_file()
    metadata_path = path.parent / "export_metadata.json"
    assert metadata_path.is_file(), "Export metadata missing; select this notebook's export."
    return path, metadata_path, json.loads(metadata_path.read_text())

def stage_offline_smoke():
    path, metadata_path, metadata = selected_export()
    script_path = path.parent / f"offline_smoke_{utc_id()}.py"
    script = r'''
import sys
import numpy as np
import torch
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
labels = ["yor", "ibo", "hau", "ego", "anw", "efi", "ibb",
          "idu", "igl", "nup", "tiv", "urh", "other"]
path = sys.argv[1]
model = AutoModelForAudioClassification.from_pretrained(
    path, local_files_only=True, trust_remote_code=False)
assert model.config.label2id == {x:i for i,x in enumerate(labels)}
assert {int(k):v for k,v in model.config.id2label.items()} == dict(enumerate(labels))
assert not getattr(model.config, "auto_map", None)
assert {p.dtype for p in model.parameters()} <= {torch.float32, torch.float16}
print("Loaded parameter dtypes:", sorted({str(p.dtype) for p in model.parameters()}))
model = model.cpu().float().eval()
fe = AutoFeatureExtractor.from_pretrained(path, local_files_only=True, trust_remote_code=False)
assert fe.sampling_rate == 16000
batch = fe([np.zeros(32000, dtype=np.float32)], sampling_rate=16000, return_tensors="pt")
with torch.inference_mode():
    logits = model(**batch).logits
assert logits.shape == (1, 13) and torch.isfinite(logits).all()
print("Offline standard-loader CPU functional smoke passed.")
'''
    script_path.write_text(script)
    env = os.environ.copy()
    env.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_DATASETS_OFFLINE="1")
    for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACEHUB_API_TOKEN"):
        env.pop(name, None)
    result = subprocess.run([sys.executable, str(script_path), str(path)],
                            env=env, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr[-5000:])
    assert result.returncode == 0, "Offline smoke failed."
    metadata["offline_smoke_passed"] = True
    metadata["smoked_core_sha256"] = {
        name: sha256_file(path / name) for name in
        ("config.json", "preprocessor_config.json", "model.safetensors")
    }
    json_write(metadata_path, metadata)

def materialize_starting_kit():
    folder = output_root() / "supplied_starting_kit"
    folder.mkdir(parents=True, exist_ok=True)
    for name, text in SUPPLIED_STARTING_KIT.items():
        path = folder / name
        if path.exists():
            assert path.read_text() == text, "Starting-kit script was modified; investigate."
        else:
            path.write_text(text)
    return folder

def stage_check_folder():
    path, metadata_path, metadata = selected_export()
    kit = materialize_starting_kit()
    result = subprocess.run(
        [sys.executable, str(kit / "check_model_folder.py"), str(path)],
        text=True, capture_output=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr[-5000:])
    metadata["supplied_checker_passed"] = (result.returncode == 0)
    metadata["supplied_kit_sha256"] = SUPPLIED_KIT_SHA256
    metadata["checked_folder_sha256"] = folder_hashes(path)
    json_write(metadata_path, metadata)
    assert result.returncode == 0, "Supplied checker failed. Fix before publishing."
    print("File/header checks passed, NOT speed/RAM/GPU compliance.")

def stage_package():
    path, _, metadata = selected_export()
    assert metadata.get("offline_smoke_passed"), "Run offline_smoke first."
    for name, digest in metadata["smoked_core_sha256"].items():
        assert sha256_file(path / name) == digest, "Core export changed since offline smoke."
    assert metadata.get("supplied_checker_passed"), "Run STAGE=check_folder first."
    assert metadata.get("checked_folder_sha256") == folder_hashes(path), \
        "Final folder changed after checker; run check_folder again."
    for key in ("LOCAL_RESOURCE_CHECKS_REVIEWED",
                "MODEL_CARD_LICENSE_REVIEWED", "PUBLIC_SNAPSHOT_CHECKED"):
        assert CFG[key], f"Manual release check not confirmed: {key}"
    assert re.fullmatch(r"[^/\s]+/[^/\s]+", CFG["PUBLIC_REPO_ID"] or "")
    assert re.fullmatch(r"[0-9a-f]{40}", CFG["PUBLIC_COMMIT"] or "")
    out = new_output("submission_packages")
    payload = {"repo": CFG["PUBLIC_REPO_ID"], "revision": CFG["PUBLIC_COMMIT"]}
    kit = materialize_starting_kit()
    completed = subprocess.run(
        [sys.executable, str(kit / "make_submission.py"),
         CFG["PUBLIC_REPO_ID"], "--revision", CFG["PUBLIC_COMMIT"]],
        cwd=str(out), text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode:
        print(completed.stderr[-5000:])
    assert completed.returncode == 0, "Supplied submission packager failed."
    with zipfile.ZipFile(out / "submission.zip") as z:
        assert z.namelist() == ["submission.json"]
        assert json.loads(z.read("submission.json")) == payload
    print("Local submission ZIP:", out / "submission.zip")
    print("No upload performed. Keep the submitted repository unchanged.")


## 8. Execute exactly one selected task

Set `STAGE` in Section 1, then **Run All from the top**. The dispatcher below is
the only task execution point. No silent "comparison skipped" or missing-global
workflow is required. If preflight finds no model, attach the actual checkpoint
files before selecting a model task.

Recommended first Kaggle sequence: **preflight → baseline → pilot → compare**.
Your Colab comparison is preserved; repeating it once on Kaggle establishes
transfer consistency. The supplied kit lacks an inference runner, so leave `RUNNER_CONFIRMED=False`.
With `EVALUATION_MODE="exploratory"`, use **reference → train_low_lr → optional
train_fixed_share** for clearly labelled local research if time permits. Ask
organisers for exact inference settings separately; do not claim deployment parity.


In [78]:
def run_stage(stage):
    tasks = {
        "preflight": stage_preflight,
        "baseline": stage_baseline,
        "pilot": stage_pilot,
        "compare": stage_compare,
        "reference": stage_reference,
        "train_low_lr": lambda: stage_train("sqrt"),
        "train_fixed_share": lambda: stage_train("fixed_share"),
        "export": stage_export,
        "offline_smoke": stage_offline_smoke,
        "check_folder": stage_check_folder,
        "package": stage_package,
    }
    assert stage in tasks, f"Unknown STAGE. Choose one of: {list(tasks)}"
    print("Running exactly one task:", stage)
    tasks[stage]()
    print("Task finished:", stage)
    print("Persist/download needed outputs from:", output_root())

run_stage(STAGE)


Running exactly one task: export
Verified epoch-1 folder: /kaggle/input/datasets/akinwandealex/dsn-2026-lid-epoch-1-checkpoint/epoch-1
SHA256: {'config.json': 'abac16bf1daa454ebbffaac33e8ecfdacd01f9875ce032a3dafc9bb89c929737', 'preprocessor_config.json': '8cdfd65ff4115423185a1512bdae100e2e0cd744f5b322417429944aaafd0827', 'model.safetensors': '3785f80820ca8bbb2d9fc1d7f43bf7e06856987ea7ccf456f2d7b424165e2a6d'}
Feature extractor: {'feature_size': 1, 'sampling_rate': 16000, 'padding_value': 0.0, 'padding_side': 'right', 'return_attention_mask': False, 'feature_extractor_type': 'Wav2Vec2FeatureExtractor', 'do_normalize': True}


Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Clean model folder: /kaggle/working/dsn_lid_review/exports/20261021T123444146370Z/model
Set CFG['EXPORT_FOLDER'] to that path for offline_smoke.
README/license review and STAGE=check_folder still required.
Task finished: export
Persist/download needed outputs from: /kaggle/working/dsn_lid_review


In [79]:
if STAGE == "train_low_lr":
    import json
    import zipfile
    from pathlib import Path
    from datetime import datetime, timezone

    root = Path(CFG["OUTPUT_ROOT"])
    available = []

    for record in sorted((root / "training").glob("*/run.json")):
        run = json.loads(record.read_text())
        candidate_name = run.get("best_candidate_dir")
        if run.get("sampling") == "sqrt" and candidate_name:
            candidate = Path(candidate_name)
            if (candidate / "model.safetensors").is_file():
                available.append((record, run, candidate))

    assert available, "No completed low-LR candidate found to back up."
    record, run, candidate = available[-1]

    backup_dir = root / "download_backups"
    backup_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    archive = backup_dir / f"low_lr_candidate_{stamp}.zip"

    with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as z:
        for file in sorted(candidate.rglob("*")):
            if file.is_file():
                z.write(file, arcname=f"model/{file.relative_to(candidate).as_posix()}")
        z.write(record, arcname="training_run.json")

    print("Backed up candidate:", candidate)
    print("Local diagnostic:", run["best_candidate_diagnostic"])
    print("DOWNLOAD THIS ZIP BEFORE ENDING THE SESSION:", archive)

In [80]:
from pathlib import Path
import json

CFG["EXPORT_FOLDER"] = (
    "/kaggle/working/dsn_lid_review/exports/"
    "20261021T123444146370Z/model"
)

folder = Path(CFG["EXPORT_FOLDER"])
assert (folder / "model.safetensors").is_file(), "New export is missing."

# Display provenance so we can confirm which checkpoint was exported.
metadata_file = folder.parent / "export_metadata.json"
assert metadata_file.is_file(), "Export metadata is missing."
print("Export provenance:")
print(json.dumps(json.loads(metadata_file.read_text()), indent=2))

run_stage("offline_smoke")
run_stage("check_folder")

Export provenance:
{
  "selected_checkpoint": "/kaggle/working/dsn_lid_review/training/20261021T122014418414Z/checkpoint-step-800",
  "dtype": "float32",
  "baseline_hashes": {
    "config.json": "abac16bf1daa454ebbffaac33e8ecfdacd01f9875ce032a3dafc9bb89c929737",
    "preprocessor_config.json": "8cdfd65ff4115423185a1512bdae100e2e0cd744f5b322417429944aaafd0827",
    "model.safetensors": "3785f80820ca8bbb2d9fc1d7f43bf7e06856987ea7ccf456f2d7b424165e2a6d"
  },
  "files_sha256": {
    "config.json": "a3856d7e98a6fbbcd79ddec4bf1aeff3359ce7a85c6b035a8ada3c33d5887522",
    "model.safetensors": "93c9ff63efc922ce83ccb831e4a3cd37710bf2dbb2aac783c8da6323e112e736",
    "preprocessor_config.json": "8cdfd65ff4115423185a1512bdae100e2e0cd744f5b322417429944aaafd0827"
  },
  "versions": {
    "torch": "2.11.0+cu128",
    "transformers": "5.16.1",
    "datasets": "4.8.5",
    "huggingface-hub": "1.29.0",
    "scikit-learn": "1.6.1",
    "soundfile": "0.14.0",
    "safetensors": "0.8.0"
  },
  "offline_smo

In [88]:
from pathlib import Path
import shutil

# The selected low-LR step-800 export.
model_dir = Path(
    "/kaggle/working/dsn_lid_review/exports/"
    "20261021T123444146370Z/model"
)
data_docs = (
    model_dir.parent
    / "source_documentation"
    / "competition_data"
)

assert (model_dir / "model.safetensors").is_file(), "Model weights missing."
assert (data_docs / "LICENSE.md").is_file(), "Source license notice missing."
assert (data_docs / "ATTRIBUTION.md").is_file(), "Source attribution missing."

attribution = (data_docs / "ATTRIBUTION.md").read_text(
    encoding="utf-8"
).strip()

readme = r"""---
pipeline_tag: audio-classification
base_model: facebook/wav2vec2-base
datasets:
- Data-Science-Nigeria/dsn-2026-lid-train
tags:
- language-identification
- wav2vec2
- nigerian-languages
---

# DSN 2026 Nigerian Language Identification

A 13-class audio-classification model fine-tuned from
`facebook/wav2vec2-base` using the official competition training split.

## Labels

Exact output order, indexed from 0 to 12:

`yor ibo hau ego anw efi ibb idu igl nup tiv urh other`

`ego` means Eggon. `other` represents speech outside the 12 target languages.

## Model and preprocessing

- Standard Hugging Face audio-classification architecture.
- Loads using `AutoModelForAudioClassification` without custom code.
- 94,571,917 parameters.
- Stored weights: fp32 safetensors.
- Input: 16 kHz mono audio.
- Feature extractor: normalization enabled, right-side zero padding,
  `return_attention_mask=False`.

No custom cropping, window aggregation, or logit calibration is embedded
in the repository.

## Training data

Official dataset:
https://huggingface.co/datasets/Data-Science-Nigeria/dsn-2026-lid-train

Pinned dataset revision:
`87cf6d4b1c7599f97f44b6e6aade86438369a2b1`

- Training: 24,774 clips.
- Validation: 4,352 clips.
- Only the training split was used for gradient updates.
- Validation was used for evaluation and checkpoint selection.
- No additional external training dataset was used.
- Pretrained initialization: `facebook/wav2vec2-base`.

The upstream corpora listed in the official dataset attribution were not
downloaded separately or used directly as additional training data.

## Training and checkpoint selection

Original fine-tuning used batch size 4, AdamW learning rate `3e-5`,
per-example sampling weights proportional to `1/sqrt(class_count)`,
and random crops up to 8 seconds.

The selected checkpoint is an 800-update continuation from epoch 1,
using a fresh optimizer at learning rate `5e-6`, the original sampler,
batch size 4, and random crops up to 8 seconds.

The convolutional feature encoder was frozen during continuation.
Training used AMP fp16; the export stores fp32 parameters.

This checkpoint comes from the recovery run completed on 2026-10-21.

## Local validation

Evaluation used full clips, batch size 1, and AMP fp16.

| Model | Represented-class diagnostic |
|---|---:|
| Original epoch 1 | 0.5684942873 |
| Selected low-LR step 800 | 0.5806749508 |

The diagnostic uses the stated fixed class shares, normalized over
classes represented in validation.

**This is not the official competition score.** Nupe has no validation
examples, so Nupe recall and F1 are unmeasured. Do not compare this
diagnostic directly with the official 0.60 ranking threshold.

Predictions into Nupe still count as errors for the true represented
classes. No Nupe-specific logit calibration was applied.

The hidden evaluator's exact preprocessing, batching, and precision were
not available in the supplied starting kit.

## Limitations

- Efik and Ibibio have zero F1 in the documented local evaluation.
- Igala and Tiv remain weak.
- Several languages have very few known training speakers.
- Nupe has one known training speaker and no validation support.
- Local results do not establish hidden-test performance on unseen
  speakers, noisy recordings, or phone-quality audio.
- Gains in some classes accompany regressions in others, including
  Hausa and Annang.
- No winning result or ranking eligibility is claimed.
- This model is not suitable for high-stakes decisions about speakers.

## Checks performed

This selected export passed:
- Standard Hugging Face offline CPU functional loading.
- The supplied starting-kit model-folder file/header checker.

These checks do not certify speed, peak RAM/GPU memory, total runtime,
or compatibility with every Transformers version.

Local environment: Transformers 5.16.1 and PyTorch 2.11.0+cu128.

## Attribution and licensing

The official dataset declares mixed per-source licensing.
Its supplied `LICENSE.md` is preserved unchanged in this repository.
The supplied attribution notice is reproduced below.

"""

readme += attribution + "\n\n"

readme += r"""### Source documentation

- Dataset:
  https://huggingface.co/datasets/Data-Science-Nigeria/dsn-2026-lid-train
- Pinned attribution:
  https://huggingface.co/datasets/Data-Science-Nigeria/dsn-2026-lid-train/blob/87cf6d4b1c7599f97f44b6e6aade86438369a2b1/ATTRIBUTION.md
- Pinned license notice:
  https://huggingface.co/datasets/Data-Science-Nigeria/dsn-2026-lid-train/blob/87cf6d4b1c7599f97f44b6e6aade86438369a2b1/LICENSE.md
- Pretrained model:
  https://huggingface.co/facebook/wav2vec2-base

The dataset card documents resampling, segmentation, label mapping,
deduplication, and speaker-based splitting. This model adds supervised
fine-tuning and continued fine-tuning. Source audio is not redistributed.

The pretrained model card declares Apache-2.0. This does not establish
a blanket Apache-2.0 license for all training-data rights or derivatives.

The dataset documentation supplies this citation:

```bibtex
@misc{omnilingualasr2025,
    title={{Omnilingual ASR}: Open-Source Multilingual Speech Recognition for 1600+ Languages},
    author={{Omnilingual ASR Team} and others},
    year={2025},
    eprint={2511.09690},
    archivePrefix={arXiv},
    primaryClass={cs.CL},
    url={https://arxiv.org/abs/2511.09690}
}
```

The supplied source notices state that some upstream audio terms are
unverified. No independent verification or blanket licensing assurance
is claimed here.
"""

# Check BOTH destinations before writing either file.
for name in ("README.md", "LICENSE.md"):
    destination = model_dir / name
    assert not destination.exists(), (
        f"{destination} already exists. Stop and inspect it before replacing."
    )

(model_dir / "README.md").write_text(readme, encoding="utf-8")
shutil.copy2(data_docs / "LICENSE.md", model_dir / "LICENSE.md")

assert (
    (model_dir / "LICENSE.md").read_bytes()
    == (data_docs / "LICENSE.md").read_bytes()
)

CFG["EXPORT_FOLDER"] = str(model_dir)

print("Documentation added. Model-folder files:")
for file in sorted(model_dir.iterdir()):
    print(" -", file.name)

# Recheck the final folder after adding documentation.
run_stage("check_folder")

Documentation added. Model-folder files:
 - LICENSE.md
 - README.md
 - config.json
 - model.safetensors
 - preprocessor_config.json
Running exactly one task: check_folder
94.6M parameters, 0.378 GB, dtypes ['F32']
PASS: the folder meets the file, size, parameter and label rules

File/header checks passed, NOT speed/RAM/GPU compliance.
Task finished: check_folder
Persist/download needed outputs from: /kaggle/working/dsn_lid_review


In [90]:
from pathlib import Path
import zipfile
import hashlib
import json

folder = Path(
    "/kaggle/working/dsn_lid_review/exports/"
    "20261021T123444146370Z/model"
)

expected = {
    "LICENSE.md",
    "README.md",
    "config.json",
    "model.safetensors",
    "preprocessor_config.json",
}

assert {p.name for p in folder.iterdir()} == expected, \
    "Unexpected or missing model-folder files."

metadata = json.loads(
    (folder.parent / "export_metadata.json").read_text()
)

# Confirm the exported model/config files still match their recorded hashes.
for name, expected_hash in metadata["files_sha256"].items():
    digest = hashlib.sha256()
    with (folder / name).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    assert digest.hexdigest() == expected_hash, f"File changed: {name}"

archive = Path("/kaggle/working/dsn_lid_final_model.zip")
assert not archive.exists(), \
    "Archive already exists; inspect/download it before recreating."

with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as z:
    for name in sorted(expected):
        z.write(folder / name, arcname=name)

print("Download this archive:", archive)
print("This is the MODEL archive, not the Codabench submission ZIP.")

AssertionError: Archive already exists; inspect/download it before recreating.

In [92]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import zipfile

folder = Path(
    "/kaggle/working/dsn_lid_review/exports/"
    "20261021T123444146370Z/model"
)
archive = Path("/kaggle/working/dsn_lid_final_model.zip")

expected = {
    "LICENSE.md",
    "README.md",
    "config.json",
    "model.safetensors",
    "preprocessor_config.json",
}

assert folder.is_dir(), "Model folder is missing."
assert {p.name for p in folder.iterdir()} == expected, \
    "Unexpected or missing model-folder files."

def sha256_stream(stream):
    digest = hashlib.sha256()
    for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
        digest.update(block)
    return digest.hexdigest()

def matches_current_model(path):
    try:
        with zipfile.ZipFile(path, "r") as z:
            names = z.namelist()
            if len(names) != len(expected) or set(names) != expected:
                return False

            for name in sorted(expected):
                with z.open(name) as saved, (folder / name).open("rb") as current:
                    if sha256_stream(saved) != sha256_stream(current):
                        return False
        return True
    except (OSError, ValueError, RuntimeError, zipfile.BadZipFile):
        return False

if archive.exists() and matches_current_model(archive):
    print("Existing ZIP verified: all five files match the current model.")
else:
    if archive.exists():
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
        archive = Path(f"/kaggle/working/dsn_lid_final_model_{stamp}.zip")

    with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as z:
        for name in sorted(expected):
            z.write(folder / name, arcname=name)

    assert matches_current_model(archive), "Archive verification failed."
    print("New ZIP created and verified.")

print("DOWNLOAD:", archive)
print("Size MB:", round(archive.stat().st_size / 1_000_000, 2))

Existing ZIP verified: all five files match the current model.
DOWNLOAD: /kaggle/working/dsn_lid_final_model.zip
Size MB: 346.09


In [93]:
import os
from pathlib import Path
from IPython.display import FileLink, display

archive = Path("/kaggle/working/dsn_lid_final_model.zip")
assert archive.is_file(), "The ZIP is missing."

previous_dir = os.getcwd()
try:
    os.chdir("/kaggle/working")
    display(FileLink(
        "dsn_lid_final_model.zip",
        result_html_prefix="Download final model ZIP: "
    ))
finally:
    os.chdir(previous_dir)

/kaggle/working/dsn_lid_final_model.zip

## Submission and persistence checklist

### Kaggle
- [ ] Actual epoch-1 checkpoint attached as a private input, hashes match.
- [ ] Input model folder is never a write target; outputs are separate.
- [ ] Internet enabled for HF/package downloads; private secret used only if needed.
- [ ] GPU identity and actual package versions recorded. No bf16 or automatic DDP.
- [ ] Save/download output artifacts before ending the session. Reattach persisted
  outputs as private input if moving sessions; update exact paths as needed.
- [ ] Set `PREVIOUS_OUTPUT_ROOT` to that attached private review-output root to
  reuse exactly matching baseline/control records. New writes still go to working.
  To export a checkpoint from an old input, select its new path and run `export`
  again; do not try to modify a read-only old export in place.
- [ ] Temporary HF cache can disappear. Reloading data is not retraining.
- [ ] Candidate checkpoints are weight snapshots, not optimizer-state resumes.

### Model and data
- [ ] Exact label order: `yor ibo hau ego anw efi ibb idu igl nup tiv urh other`.
- [ ] `ego` means Eggon. Keep `other` and Nupe as output classes.
- [ ] No Nupe-specific fitted bias; Nupe F1/recall remain unmeasured on validation.
- [ ] No direct training on the three prohibited sources.
- [ ] No test-set tuning, test access or cross-team artifacts.
- [ ] Explain official data, pretrained initialization and any permitted external
  data in README; retain applicable attribution/license terms.

### Official release checks (not certified by this notebook)
- [ ] `STAGE="check_folder"` passes on the final model folder using the supplied
  script. Re-run after any file change, including README/license edits.
- [ ] Standard `AutoModelForAudioClassification`, no custom code or `auto_map`.
- [ ] fp32/fp16 safetensors, no `.bin`, `.pt`, `.py` or training diagnostics in repo.
- [ ] Parameters <=350M; final repo <=1.5 GB.
- [ ] Local GPU memory **and RAM** checks reviewed. Official <=4 GB compliance
  is ultimately measured by organisers; a folder-checker pass cannot prove it.
- [ ] Whole-pipeline average <=0.5 s/clip under official warmed-up,
  median-of-three procedure; loading + full hidden run <=30 min.
- [ ] Local timings are not official fixed-hardware measurements.
- [ ] Offline inference on real 2–30-second clips. Runner preprocessing either
  actually confirmed or explicitly documented as unknown; do not falsely attest parity.
- [ ] Public exact pinned revision loads without relying on account authentication.
- [ ] Only explicitly permitted model/preprocessor/README/license files.
- [ ] Submitted repo unchanged after submission; conservatively use a new repo
  for a later candidate. Pinned commit alone does not waive the freeze rule.
- [ ] ZIP contains only root-level `submission.json` with `repo` and `revision`.
- [ ] Confirm exact deadline hour/timezone and submission allowance on Codabench.

The supplied shares total 0.999 because of stated precision. Check the starting
kit's exact full metric implementation. The incomplete validation diagnostic is
**not** the official language-weighted F1 or final efficiency-adjusted score.

Paste back Kaggle preflight, fingerprint/model inspection, then pilot/compare
results. Send no secret values. No winning result or ranking eligibility is promised.

Rules reference: https://www.codabench.org/competitions/18424/
This notebook follows the user's pasted brief; no live rule-page verification is claimed.
